# 점검 1 — 좌회전 운영 확인 + 상충유형별 후보 사건 수 (20곳 전부)

**목적 두 가지**
1. **좌회전 운영(보호 vs 비보호)**: Zheng & Sayed (2019, TR-C)의 crossing 상충은 좌회전 차량과 그 경로를 가로지르는 직진 차량이 교차로 안에 **동시에** 있어야 생긴다. 보호신호(화살표)만으로 운영되면 이 표본이 거의 없다.
2. **상충유형별 후보 사건 수**: 후미추돌(Zheng·Sayed·Essa 2019, AAP 방식), 좌회전 crossing, 우회전 합류 각각의 후보 쌍이 지점마다 몇 건인지 센다. 논문에 "왜 이 유형을 택했는가"를 표로 보여주기 위한 근거다. 메인 논문들은 자기 지점에 있는 유형을 골라 사전선별(TTC < 4초 등)만 했지만, 우리는 20곳 운영을 모르니 먼저 세어야 한다.

**방법**: 신호운영 자료 없이 궤적만으로 판단한다.
1. `Road_Section`의 N(node)으로 진입·진출 접근로를 잡는다. 교차로 내부는 라벨이 없어서(README) 그 빈 구간이 "교차로 안에 있던 시간"이 된다.
2. 접근로 위치를 궤적 좌표(`Local_X`, `Local_Y`, 미터)로 계산해 맞은편 짝을 정한다. 3지(T자) 가지 접근로는 맞은편이 없다.
3. 회전각으로 직진/좌회전/우회전/유턴을 분류한다.
4. 유형별 후보 쌍을 센다.
   - **좌회전 crossing (opp)**: 좌회전 vs 맞은편 접근로에서 온 직진. 4지의 핵심. Zheng & Sayed가 본 쌍.
   - **좌회전 crossing (cross)**: 좌회전 vs 진출 접근로에서 온 직진. 3지 가지의 핵심. 4지에서는 신호위반 아니면 0이어야 함.
   - **우회전 합류**: 우회전 vs 우회전이 합류하는 도로의 직진. 한국은 적색 시 우회전이 허용돼 흔한 유형. 메인 논문 선례는 없음.
   - **후미추돌**: 같은 접근로·같은 차로에서 앞차가 아직 있을 때 뒤차가 들어온 연속 차량쌍. Zheng·Sayed·Essa 2019 §3.2의 "직진차로 연속 차량쌍"에 해당.

**좌회전 운영 판정 논리(임의 퍼센트가 아니라 신호운영 구조에서 나오는 값)**

| 운영 | 좌회전과 직진이 같이 있을 수 있는 시간 | 겹침 비율의 기대치 |
|---|---|---|
| 보호좌회전만 | 현시 전환(황색·전적색) 몇 초 + 신호위반 | 주기(2~3분) 중 몇 초뿐이라 0에 가까움 |
| 비보호 또는 겸용 | 직진 녹색 내내 | 맞은편에 차가 지나가면 좌회전이 기다리므로 대부분 겹침 |

두 경우가 한 자릿수 이상 차이 나므로 특정 숫자 하나가 결론을 가르지 않는다. (a) 겹침 비율의 크기, (b) 겹친 사례가 짧은 시간에 몰려 있는지, (c) 시간축 그림에서 두 이동류가 번갈아 블록으로 나오는지, 이 셋을 같이 본다.

**대상**: 20곳 전부 (4지 15곳, 3지 5곳). 먼저 K 한 세션으로 절차를 확인한 뒤 확장한다. 원본 데이터는 읽기만 한다.

In [ ]:
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
plt.rcParams["font.family"] = "Malgun Gothic"   # 한글 축 이름 깨짐 방지 (Windows 기본 폰트)
plt.rcParams["axes.unicode_minus"] = False        # 한글 폰트에서 음수 부호(-) 깨짐 방지
import pandas as pd

DATA_DIR = Path('C:/Users/123/Documents/(송도) 교통 연구 논문/data/raw')
DAYS = ['2022-10-04', '2022-10-05', '2022-10-06', '2022-10-07']
SESSIONS = ['AM1', 'AM2', 'AM3', 'AM4', 'AM5', 'PM1', 'PM2', 'PM3', 'PM4', 'PM5']
INTERSECTIONS = list('ABCEFGHIJKLMNOPQRSTU')   # 20곳
FOUR_LEG = list('ABFHIJKLMNOPRST')             # 4지 15곳
THREE_LEG = list('CEGQU')                      # 3지 5곳

SITE = 'K'
print('대상 교차로:', SITE, '| 구조:', '4지' if SITE in FOUR_LEG else '3지')

## 1. 파일 하나 읽기

쓰는 컬럼: `Vehicle_ID`, `Local_Time`, `Road_Section`, `Lane_Number`, `Local_X`, `Local_Y`. `Road_Section`은 `N_G` 형식이라 앞의 N을 node로 쓴다. `Lane_Number`는 후미추돌 쌍(같은 차로)을 만들 때 쓴다.

In [ ]:
def load_file(site, day, session):
    f = DATA_DIR / f'{day}_{site}' / f'{day}_{site}_{session}.csv'
    df = pd.read_csv(f, usecols=['Vehicle_ID', 'Local_Time', 'Road_Section', 'Lane_Number', 'Local_X', 'Local_Y'],
                     dtype={'Local_Time': str, 'Road_Section': str})
    df['t'] = pd.to_timedelta(df['Local_Time']).dt.total_seconds()
    df['N'] = df['Road_Section'].str.split('_').str[0]      # node. 내부(라벨 없음)는 NaN
    df = df.sort_values(['Vehicle_ID', 't']).reset_index(drop=True)
    return df

df = load_file(SITE, DAYS[0], 'AM1')
print('행:', len(df), '| 차량:', df['Vehicle_ID'].nunique(), '| 내부(라벨 없음) 행:', int(df['N'].isna().sum()))
print('node 종류:', sorted(df['N'].dropna().unique()))

## 2. 접근로 위치와 맞은편 짝

각 node에 라벨된 궤적점들의 평균 좌표를 그 접근로의 위치로, 그 평균을 교차로 중심으로 잡는다. 중심에서 본 방위각이 180도에 가까운 쌍이 맞은편이다. 180도에서 45도 이상 벗어나면 맞은편 없음으로 둔다(3지의 가지).

확인 포인트: 방위각(동쪽 0도, 반시계 +)이 4지면 대략 90도 간격인지, 맞은편 짝이 정사영상과 맞는지.

In [ ]:
def node_geometry(df):
    lab = df.dropna(subset=['N'])
    cent = lab.groupby('N')[['Local_X', 'Local_Y']].mean()
    center = cent.mean()
    vec = cent - center
    cent['bearing_deg'] = np.degrees(np.arctan2(vec['Local_Y'], vec['Local_X'])).round(1)
    nodes = list(cent.index)
    opp = {}
    for a in nodes:
        best, best_gap = None, 999
        for b in nodes:
            if b == a:
                continue
            gap = abs(((cent.loc[b, 'bearing_deg'] - cent.loc[a, 'bearing_deg']) % 360) - 180)
            if gap < best_gap:
                best, best_gap = b, gap
        opp[a] = best if best_gap <= 45 else None
    cent['opposite'] = pd.Series(opp)
    return cent, center

cent, center = node_geometry(df)
print('교차로 중심 (Local_X, Local_Y):', center.round(1).to_dict())
cent

In [ ]:
# (선택) 접근로 배치 그림. 정사영상과 비교용
try:
    import matplotlib.pyplot as plt
    fig, ax = plt.subplots(figsize=(5, 5))
    ax.scatter(cent['Local_X'], cent['Local_Y'], s=80)
    for n, r in cent.iterrows():
        ax.annotate(f'N{n}', (r['Local_X'], r['Local_Y']), textcoords='offset points', xytext=(6, 6))
        if r['opposite'] is not None:
            o = cent.loc[r['opposite']]
            ax.plot([r['Local_X'], o['Local_X']], [r['Local_Y'], o['Local_Y']], 'k--', lw=0.8)
    ax.scatter([center['Local_X']], [center['Local_Y']], marker='x', c='r')
    ax.set_aspect('equal'); ax.set_title(f'{SITE}: node 위치와 맞은편 짝 (북쪽이 위)')
    plt.show()
except ImportError:
    print('matplotlib 없음 - 그림 생략')

## 3. 차량별 이동류 분류와 교차로 내부 체류시간

- 진입 node = 처음 라벨된 node, 진출 node = 마지막 라벨된 node. 둘 다 있고 서로 다른 차량만 쓴다(한쪽만 찍힌 조각 궤적은 제외).
- 회전각: 진입 방향(접근로에서 중심으로)과 진출 방향(중심에서 접근로로) 사이의 부호 있는 각. 좌표가 미터 단위에 북쪽이 +y라서 반시계(+)가 좌회전이다.
  - |각| ≤ 45도: 직진 / 45~135도: 좌회전 / −135~−45도: 우회전 / 그 밖: 유턴·기타
- 교차로 안에 있던 시간: `t_in` = 진입 접근로 라벨이 마지막으로 붙은 시각, `t_out` = 진출 접근로 라벨이 처음 붙은 시각.

In [ ]:
def classify_vehicles(df, cent, center):
    lab = df.dropna(subset=['N'])
    first = lab.groupby('Vehicle_ID')[['N', 't']].first().rename(columns={'N': 'entry', 't': 't_entry_first'})
    last = lab.groupby('Vehicle_ID')[['N', 't']].last().rename(columns={'N': 'exit', 't': 't_exit_last'})
    v = first.join(last)
    v = v[v['entry'] != v['exit']].copy()

    m = lab.merge(v[['exit', 't_entry_first']], left_on='Vehicle_ID', right_index=True)
    m = m[(m['N'] == m['exit']) & (m['t'] > m['t_entry_first'])]
    v['t_out'] = m.groupby('Vehicle_ID')['t'].min()
    m = lab.merge(v[['entry', 't_out']], left_on='Vehicle_ID', right_index=True)
    m = m[(m['N'] == m['entry']) & (m['t'] < m['t_out'])]
    v['t_in'] = m.groupby('Vehicle_ID')['t'].max()
    v = v.dropna(subset=['t_in', 't_out'])

    def turn_angle(a, b):
        h = center[['Local_X', 'Local_Y']].to_numpy() - cent.loc[a, ['Local_X', 'Local_Y']].to_numpy()
        e = cent.loc[b, ['Local_X', 'Local_Y']].to_numpy() - center[['Local_X', 'Local_Y']].to_numpy()
        return float(np.degrees(np.arctan2(h[0] * e[1] - h[1] * e[0], h[0] * e[0] + h[1] * e[1])))
    v['angle'] = [turn_angle(a, b) for a, b in zip(v['entry'], v['exit'])]
    v['move'] = pd.cut(v['angle'], bins=[-180, -135, -45, 45, 135, 180],
                       labels=['uturn_other', 'right', 'through', 'left', 'uturn_other'], ordered=False)
    v['dwell_s'] = (v['t_out'] - v['t_in']).round(2)
    return v

veh = classify_vehicles(df, cent, center)
print('분류된 차량:', len(veh))
print(pd.crosstab(veh['entry'], veh['move']))
print('\n교차로 내부 체류시간(초) 요약:'); print(veh.groupby('move', observed=True)['dwell_s'].describe().round(1))

확인 포인트: 직진이 가장 많고, 좌회전·우회전이 접근로마다 수십 대씩 있으면 정상. 좌회전 체류시간이 직진보다 길고 분산이 크면(교차로 안에서 대기) 비보호의 힌트, 짧고 균일하면 보호의 힌트. 다음 절에서 직접 센다.

## 4. 유형별 후보 쌍 세기

접근로 a마다 다음을 센다. `n_pairs_*`는 겹친 쌍의 총수로, 그 유형의 상충 후보 **상한**이다(실제 상충은 이 중 TTC·PET가 작은 것만).

| 열 | 유형 | 상대 차량 |
|---|---|---|
| `n_left`, `share_opp`, `n_pairs_opp` | 좌회전 crossing (4지 핵심) | 맞은편 접근로 opp(a)에서 온 직진 |
| `share_cross`, `n_pairs_cross` | 좌회전 crossing (3지 가지 핵심) | 좌회전이 빠져나가는 접근로 b에서 온 직진 |
| `n_right`, `n_pairs_right` | 우회전 합류 | 우회전이 합류하는 도로의 직진(진출 접근로 c의 맞은편에서 c로 가는 직진) |
| `n_rear_all`, `n_rear_through` | 후미추돌 | 같은 접근로·같은 차로의 연속 차량쌍. `through`는 직진이 우세한 차로만 |

좌회전·우회전 쌍은 "교차로 내부 시간구간이 겹침"으로, 후미추돌 쌍은 "앞차가 아직 그 차로에 있을 때 뒤차가 그 차로에 들어옴"으로 센다. 후미추돌은 Zheng·Sayed·Essa 2019 §3.2의 "직진차로의 연속 차량쌍(전용 회전차로 제외)"에 맞추려고 직진 우세 차로(분류된 차량 중 직진이 60% 이상)만 따로 센다.

In [ ]:
def _overlap(L, T):
    if len(L) == 0 or len(T) == 0:
        return 0, np.nan, 0
    li, lo = L['t_in'].to_numpy()[:, None], L['t_out'].to_numpy()[:, None]
    ti, to = T['t_in'].to_numpy()[None, :], T['t_out'].to_numpy()[None, :]
    ov = (ti < lo) & (to > li)
    return int(ov.any(axis=1).sum()), round(float(ov.any(axis=1).mean()), 3), int(ov.sum())

def rear_end_pairs(df, veh):
    """접근로·차로별 연속 차량쌍. 뒤차가 차로에 들어올 때 앞차가 아직 그 차로에 있으면 후보로 센다."""
    lab = df.dropna(subset=['Road_Section', 'Lane_Number'])
    pres = (lab.groupby(['Road_Section', 'Lane_Number', 'Vehicle_ID'])['t']
               .agg(t_first='min', t_last='max').reset_index())
    pres = pres.sort_values(['Road_Section', 'Lane_Number', 't_first'])
    g = pres.groupby(['Road_Section', 'Lane_Number'])
    pres['next_first'] = g['t_first'].shift(-1)
    pres['is_pair'] = pres['next_first'].notna() & (pres['next_first'] < pres['t_last'])
    # 차로별 직진 우세 여부 (분류된 차량 기준)
    mv = veh['move'].astype(str)
    pres['move'] = pres['Vehicle_ID'].map(mv)
    lane_stat = (pres.dropna(subset=['move']).groupby(['Road_Section', 'Lane_Number'])['move']
                     .agg(n='size', through_share=lambda s: (s == 'through').mean()).reset_index())
    lane_stat['through_lane'] = (lane_stat['n'] >= 5) & (lane_stat['through_share'] >= 0.6)
    pres = pres.merge(lane_stat[['Road_Section', 'Lane_Number', 'through_lane']], on=['Road_Section', 'Lane_Number'], how='left')
    pres['through_lane'] = pres['through_lane'].fillna(False)
    pres['approach'] = pres['Road_Section'].str.split('_').str[0]
    out = (pres.groupby('approach')
               .agg(n_rear_all=('is_pair', 'sum'),
                    n_rear_through=('is_pair', lambda s: int((s & pres.loc[s.index, 'through_lane']).sum())),
                    n_through_lanes=('through_lane', lambda s: int(pres.loc[s.index].drop_duplicates(['Road_Section', 'Lane_Number'])['through_lane'].sum())))
               .reset_index())
    out['n_rear_all'] = out['n_rear_all'].astype(int)
    return out, lane_stat

def coexistence(veh, cent, df=None):
    rows = []
    thr = veh[veh['move'] == 'through']
    for a in cent.index:
        L = veh[(veh['entry'] == a) & (veh['move'] == 'left')]
        R = veh[(veh['entry'] == a) & (veh['move'] == 'right')]
        o = cent.loc[a, 'opposite']
        b = L['exit'].mode().iloc[0] if len(L) else None       # 좌회전 진출 접근로
        c = R['exit'].mode().iloc[0] if len(R) else None       # 우회전 진출 접근로
        oc = cent.loc[c, 'opposite'] if c is not None else None  # 우회전이 합류하는 도로의 직진은 opp(c)에서 c로
        T_opp = thr[thr['entry'] == o] if o is not None else thr.iloc[0:0]
        T_cross = thr[thr['entry'] == b] if b is not None else thr.iloc[0:0]
        T_merge = thr[thr['entry'] == oc] if oc is not None else thr.iloc[0:0]
        n1, s1, p1 = _overlap(L, T_opp)
        n2, s2, p2 = _overlap(L, T_cross)
        n3, s3, p3 = _overlap(R, T_merge)
        rows.append({'approach': a, 'opposite': o, 'left_exit': b, 'n_left': len(L),
                     'n_opp_through': len(T_opp), 'n_left_overlap_opp': n1, 'share_opp': s1, 'n_pairs_opp': p1,
                     'n_cross_through': len(T_cross), 'n_left_overlap_cross': n2, 'share_cross': s2, 'n_pairs_cross': p2,
                     'right_exit': c, 'n_right': len(R), 'n_merge_through': len(T_merge), 'share_right': s3, 'n_pairs_right': p3})
    co = pd.DataFrame(rows)
    if df is not None:
        rear, _ = rear_end_pairs(df, veh)
        co = co.merge(rear, on='approach', how='left')
    return co

co = coexistence(veh, cent, df)
co

In [ ]:
# 유형별 후보 쌍 수 한눈에 (이 세션)
summary = pd.DataFrame({
    '좌회전 crossing (opp)': [co['n_pairs_opp'].sum()],
    '좌회전 crossing (cross)': [co['n_pairs_cross'].sum()],
    '우회전 합류': [co['n_pairs_right'].sum()],
    '후미추돌 (전체 차로)': [co['n_rear_all'].sum()],
    '후미추돌 (직진 우세 차로)': [co['n_rear_through'].sum()],
}, index=[f'{SITE} {DAYS[0]} AM1'])
summary

In [ ]:
# 겹친 사례를 눈으로 확인 (좌회전 vs 맞은편 직진, 앞 5개)
shown = 0
thr = veh[veh['move'] == 'through']
for a in cent.index:
    o = cent.loc[a, 'opposite']
    if o is None:
        continue
    L = veh[(veh['entry'] == a) & (veh['move'] == 'left')]
    T = thr[thr['entry'] == o]
    for vid, r in L.iterrows():
        hit = T[(T['t_in'] < r['t_out']) & (T['t_out'] > r['t_in'])]
        if len(hit):
            print(f'좌회전 {vid} (N{a}->N{r["exit"]}) 내부 {r["t_in"]:.1f}~{r["t_out"]:.1f}s | 겹친 맞은편 직진:',
                  [(int(i), round(h.t_in, 1), round(h.t_out, 1)) for i, h in hit.iterrows()][:5])
            shown += 1
        if shown >= 5:
            break
    if shown >= 5:
        break
if shown == 0:
    print('이 세션에는 맞은편 직진과 겹친 좌회전이 없음')

## 5. 임계값 없는 확인: 시간축 그림

좌회전이 가장 많은 접근로 하나를 골라, 세션 30분 동안 좌회전 차량의 진입 시각(t_in)과 맞은편 직진·교차 직진의 진입 시각을 한 축에 찍는다.
- 보호운영: 좌회전 무리와 맞은편 직진 무리가 **번갈아 블록**으로 나오고 섞이지 않는다.
- 비보호·겸용: 두 무리가 **섞여** 나온다.

In [ ]:
try:
    import matplotlib.pyplot as plt
    a = co.sort_values('n_left', ascending=False).iloc[0]['approach']
    o, b = cent.loc[a, 'opposite'], co.set_index('approach').loc[a, 'left_exit']
    L = veh[(veh['entry'] == a) & (veh['move'] == 'left')]
    T_opp = veh[(veh['entry'] == o) & (veh['move'] == 'through')] if o is not None else veh.iloc[0:0]
    T_cross = veh[(veh['entry'] == b) & (veh['move'] == 'through')]
    t0 = veh['t_in'].min()
    fig, ax = plt.subplots(figsize=(12, 3))
    ax.eventplot([(L['t_in'] - t0) / 60, (T_opp['t_in'] - t0) / 60, (T_cross['t_in'] - t0) / 60],
                 lineoffsets=[2, 1, 0], linelengths=0.8, colors=['red', 'blue', 'gray'])
    ax.set_yticks([2, 1, 0]); ax.set_yticklabels([f'좌회전 N{a}->N{b}', f'맞은편 직진 N{o}', f'교차 직진 N{b}'])
    ax.set_xlabel('세션 시작 후 경과(분)'); ax.set_title(f'{SITE} 접근로 N{a}: 좌회전 vs 직진의 교차로 진입 시각')
    plt.tight_layout(); plt.show()
except ImportError:
    print('matplotlib 없음 - 그림 생략')

## 6. 같은 교차로의 40개 세션으로 확장

4일 × 10세션을 다 돌려 접근로별로 합산한다. 접근로 위치는 세션마다 다시 계산한다.

In [ ]:
SUM_COLS = ['n_left', 'n_opp_through', 'n_left_overlap_opp', 'n_pairs_opp',
            'n_cross_through', 'n_left_overlap_cross', 'n_pairs_cross',
            'n_right', 'n_merge_through', 'n_pairs_right', 'n_rear_all', 'n_rear_through']

def run_site(site, days=DAYS, sessions=SESSIONS, verbose=True):
    out = []
    for day in days:
        for s in sessions:
            f = DATA_DIR / f'{day}_{site}' / f'{day}_{site}_{s}.csv'
            if not f.exists():
                print('파일 없음:', f.name); continue
            d = load_file(site, day, s)
            c, ctr = node_geometry(d)
            v = classify_vehicles(d, c, ctr)
            r = coexistence(v, c, d)
            r.insert(0, 'session', s); r.insert(0, 'day', day); r.insert(0, 'site', site)
            out.append(r)
        if verbose:
            print(site, day, '완료')
    res = pd.concat(out, ignore_index=True)
    agg = res.groupby(['site', 'approach'], dropna=False).agg(
        opposite=('opposite', 'first'), left_exit=('left_exit', 'first'), files=('session', 'count'),
        **{c_: (c_, 'sum') for c_ in SUM_COLS}).reset_index()
    agg['share_opp'] = (agg['n_left_overlap_opp'] / agg['n_left']).round(3)
    agg['share_cross'] = (agg['n_left_overlap_cross'] / agg['n_left']).round(3)
    agg.insert(1, 'structure', np.where(agg['site'].isin(FOUR_LEG), '4지', '3지'))
    return res, agg

res_site, agg_site = run_site(SITE)
agg_site

In [ ]:
# 세션별 share_opp (시간대·요일에 따라 운영이 다른지) + 유형별 후보 쌍 합계
pv = res_site.pivot_table(index=['day', 'session'], columns='approach', values='share_opp')
display(pv.round(2))
print('유형별 후보 쌍 합계 (40세션):')
print(agg_site[['n_pairs_opp', 'n_pairs_cross', 'n_pairs_right', 'n_rear_all', 'n_rear_through']].sum())

OUT_DIR = Path.cwd() / 'outputs'
OUT_DIR.mkdir(exist_ok=True)
res_site.to_csv(OUT_DIR / f'02_좌회전운영_{SITE}_세션별.csv', index=False, encoding='utf-8-sig')
agg_site.to_csv(OUT_DIR / f'02_좌회전운영_{SITE}_집계.csv', index=False, encoding='utf-8-sig')
print('저장:', OUT_DIR)

## 7. 20곳 전부 (먼저 첫날만, 이후 4일 전체)

K가 해석되면 20곳을 모두 돈다. 지점 × 접근로별 share와 유형별 후보 쌍 수를 한 표로 모은다. 첫날만 해도 200파일이라 시간이 걸린다.

In [ ]:
# all_res, all_agg = [], []
# for site in INTERSECTIONS:
#     r, a = run_site(site, days=DAYS[:1], verbose=False)
#     all_res.append(r); all_agg.append(a); print(site, '완료')
# all_res = pd.concat(all_res, ignore_index=True); all_agg = pd.concat(all_agg, ignore_index=True)
# all_agg.to_csv(OUT_DIR / '02_좌회전운영_20곳_첫날.csv', index=False, encoding='utf-8-sig')
# display(all_agg.pivot_table(index=['structure', 'site'], columns='approach', values='share_opp').round(2))
# by_site = all_agg.groupby(['structure', 'site'])[['n_pairs_opp', 'n_pairs_cross', 'n_pairs_right', 'n_rear_all', 'n_rear_through']].sum()
# display(by_site)   # 지점별 유형별 후보 쌍 수 = "왜 이 유형인가" 표의 재료

## 8. 결과를 어떻게 읽을 것인가

**좌회전 운영**
- 4지: `share_opp`가 핵심. 0에 가깝고 겹친 사례가 전환 순간에 몰려 있으며 시간축 그림에서 블록이 번갈아 나오면 보호좌회전. `share_cross`는 신호위반 외엔 0이어야 정상.
- 3지 가지(맞은편 없음): `share_cross`가 핵심. 본선 접근로는 `share_opp`.

**상충유형 선택**: 지점별 `n_pairs_*` 합계 표가 "왜 이 유형인가"의 근거다.
- 보호운영이면 `n_pairs_opp`는 작고 `n_rear_through`는 크게 나올 것이다. 그러면 후미추돌(Zheng·Sayed·Essa 2019 §3.2)로 확정하고, 좌회전 crossing과 우회전 합류는 후보 수를 표로 남긴 채 후속연구로 둔다.
- 어느 지점·접근로에서 `share_opp`가 크게 나오면 비보호 또는 겸용. 그곳은 crossing(Zheng & Sayed 2019 §4)이 가능하고 `n_pairs_opp`가 표본 상한이다. 주분석은 그래도 후미추돌 20곳으로 가고, crossing은 그 지점의 부가분석으로 둔다(유형은 논문당 하나).
- 우회전 합류는 메인 논문 선례가 없으므로 이번 논문에서는 세어두기만 한다.

**주의**
- 궤적 분절(원논문 Appendix D) 때문에 한쪽 접근로만 찍힌 차량은 이동류 분류에서 제외됐다. 비율에는 큰 영향이 없지만 대수는 실제보다 적을 수 있다. 후미추돌 쌍은 분류와 무관하게 차로 안 존재 시간으로 세므로 이 영향이 작다.
- `n_rear_*`는 "같이 있었던 연속 쌍"의 수이지 상충 수가 아니다. 실제 상충은 이 중 TTC < 4초 등 사전선별을 통과한 것만이고, 그 단계는 다음 노트북에서 한다.
- 논문 Data 절의 "보호/비보호 운영" 서술은 로드뷰(좌회전 화살표 신호등, 비보호 표지)로 따로 확인해 붙인다.